In [5]:
from pathlib import Path
import pandas as pd

# =========================
# CONFIG DATASET
# =========================

BASE_DIR = Path("/media/spell/Spell-lab/Lidar/A.Raw Dataset")

ACTIVITIES = ["Bungkuk", "Duduk", "Jongkok", "Jatuh"]

SUBJECTS = [
    "Adelia",
    "Afi",
    "Aswangga",
    "Bustan",
    "Dilia",
    "Eldivo",
    "Fathir",
    "Lina",
    "Manda",
    "Miftah",
    "Teguh",
    "Tsamara",
]

EXPECTED_FILES = [f"{i}.Csv" for i in range(1, 10)]

REQUIRED_COLUMNS = ["Timestamp", "X", "Y", "Z", "Reflectivity", "Tag"]

expected_total = len(ACTIVITIES) * len(SUBJECTS) * len(EXPECTED_FILES)

print("Base directory:", BASE_DIR)
print("Base directory exists:", BASE_DIR.exists())
print("Jumlah activity:", len(ACTIVITIES), ACTIVITIES)
print("Jumlah subject:", len(SUBJECTS), SUBJECTS)
print("File per subject per activity:", EXPECTED_FILES)
print("Expected total CSV:", expected_total)

Base directory: /media/spell/Spell-lab/Lidar/A.Raw Dataset
Base directory exists: True
Jumlah activity: 4 ['Bungkuk', 'Duduk', 'Jongkok', 'Jatuh']
Jumlah subject: 12 ['Adelia', 'Afi', 'Aswangga', 'Bustan', 'Dilia', 'Eldivo', 'Fathir', 'Lina', 'Manda', 'Miftah', 'Teguh', 'Tsamara']
File per subject per activity: ['1.Csv', '2.Csv', '3.Csv', '4.Csv', '5.Csv', '6.Csv', '7.Csv', '8.Csv', '9.Csv']
Expected total CSV: 432


In [6]:
# =========================
# AUDIT FILE STRUCTURE
# =========================

records = []

for activity in ACTIVITIES:
    activity_dir = BASE_DIR / activity
    
    for subject in SUBJECTS:
        subject_dir = activity_dir / subject
        csv_dir = subject_dir / "CSV"
        
        for filename in EXPECTED_FILES:
            file_path = csv_dir / filename
            
            records.append({
                "activity": activity,
                "subject": subject,
                "trial_file": filename,
                "activity_dir_exists": activity_dir.exists(),
                "subject_dir_exists": subject_dir.exists(),
                "csv_dir_exists": csv_dir.exists(),
                "file_exists": file_path.exists(),
                "file_path": str(file_path),
            })

audit_df = pd.DataFrame(records)

total_found = audit_df["file_exists"].sum()
total_missing = expected_total - total_found

print("===== AUDIT STRUKTUR DATASET =====")
print("Expected total files:", expected_total)
print("Found files         :", total_found)
print("Missing files       :", total_missing)
print("Completion          : {:.2f}%".format(100 * total_found / expected_total))

print("\nMissing files preview:")
display(audit_df[audit_df["file_exists"] == False].head(30))

print("\nSummary by activity:")
display(
    audit_df.groupby("activity")["file_exists"]
    .agg(expected="count", found="sum")
    .assign(missing=lambda x: x["expected"] - x["found"])
    .reset_index()
)

===== AUDIT STRUKTUR DATASET =====
Expected total files: 432
Found files         : 432
Missing files       : 0
Completion          : 100.00%

Missing files preview:


,activity,subject,trial_file,activity_dir_exists,subject_dir_exists,csv_dir_exists,file_exists,file_path



Summary by activity:


,activity,expected,found,missing
0,Bungkuk,108,108,0
1,Duduk,108,108,0
2,Jatuh,108,108,0
3,Jongkok,108,108,0


In [7]:
# =========================
# AUDIT READABILITY & COLUMNS
# =========================

read_records = []

existing_files = audit_df[audit_df["file_exists"] == True].copy()

for idx, row in existing_files.iterrows():
    file_path = Path(row["file_path"])
    
    status = {
        "activity": row["activity"],
        "subject": row["subject"],
        "trial_file": row["trial_file"],
        "file_path": row["file_path"],
        "file_size_bytes": None,
        "is_empty_file": None,
        "can_read": False,
        "num_rows": None,
        "num_cols": None,
        "columns": None,
        "missing_required_columns": None,
        "has_required_columns": False,
        "error": None,
    }
    
    try:
        file_size = file_path.stat().st_size
        status["file_size_bytes"] = file_size
        status["is_empty_file"] = file_size == 0
        
        if file_size == 0:
            status["error"] = "Empty file"
        else:
            # Baca sedikit dulu agar audit cepat
            temp_df = pd.read_csv(file_path, nrows=5)
            
            status["can_read"] = True
            status["num_cols"] = len(temp_df.columns)
            status["columns"] = list(temp_df.columns)
            
            missing_cols = [col for col in REQUIRED_COLUMNS if col not in temp_df.columns]
            status["missing_required_columns"] = missing_cols
            status["has_required_columns"] = len(missing_cols) == 0
            
            # Hitung jumlah baris total
            # Lebih ringan daripada load full dataframe
            with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
                num_lines = sum(1 for _ in f)
            
            # dikurangi header
            status["num_rows"] = max(num_lines - 1, 0)
            
    except Exception as e:
        status["error"] = str(e)
    
    read_records.append(status)

read_audit_df = pd.DataFrame(read_records)

print("===== AUDIT READABILITY & KOLOM =====")
print("Existing files checked:", len(read_audit_df))
print("Readable files        :", read_audit_df["can_read"].sum())
print("Unreadable files      :", (~read_audit_df["can_read"]).sum())
print("Empty files           :", read_audit_df["is_empty_file"].sum())
print("Files with required columns:", read_audit_df["has_required_columns"].sum())

print("\nUnreadable / empty / missing columns preview:")
problem_df = read_audit_df[
    (read_audit_df["can_read"] == False) |
    (read_audit_df["is_empty_file"] == True) |
    (read_audit_df["has_required_columns"] == False)
]

display(problem_df.head(30))

print("\nRow count statistics:")
display(read_audit_df["num_rows"].describe())

===== AUDIT READABILITY & KOLOM =====
Existing files checked: 432
Readable files        : 432
Unreadable files      : 0
Empty files           : 0
Files with required columns: 432

Unreadable / empty / missing columns preview:


,activity,subject,trial_file,file_path,file_size_bytes,is_empty_file,can_read,num_rows,num_cols,columns,missing_required_columns,has_required_columns,error



Row count statistics:


count    4.320000e+02
mean     1.236301e+06
std      1.435009e+05
min      8.843530e+05
25%      1.137313e+06
50%      1.210801e+06
75%      1.312681e+06
max      2.270785e+06
Name: num_rows, dtype: float64

In [8]:
# =========================
# SUMMARY PER ACTIVITY & SUBJECT
# =========================

# Merge audit struktur + audit read
merged_df = audit_df.merge(
    read_audit_df[
        [
            "activity", "subject", "trial_file",
            "file_size_bytes", "is_empty_file", "can_read",
            "num_rows", "num_cols", "missing_required_columns",
            "has_required_columns", "error"
        ]
    ],
    on=["activity", "subject", "trial_file"],
    how="left"
)

summary_subject = (
    merged_df
    .groupby(["activity", "subject"])
    .agg(
        expected_files=("trial_file", "count"),
        found_files=("file_exists", "sum"),
        readable_files=("can_read", "sum"),
        empty_files=("is_empty_file", "sum"),
        files_with_required_columns=("has_required_columns", "sum"),
        total_rows=("num_rows", "sum"),
    )
    .reset_index()
)

summary_subject["missing_files"] = summary_subject["expected_files"] - summary_subject["found_files"]
summary_subject["completion_percent"] = 100 * summary_subject["found_files"] / summary_subject["expected_files"]

print("===== SUMMARY PER ACTIVITY & SUBJECT =====")
display(summary_subject)

print("\nSubject/activity yang belum lengkap:")
display(
    summary_subject[
        (summary_subject["missing_files"] > 0) |
        (summary_subject["readable_files"] < summary_subject["expected_files"]) |
        (summary_subject["files_with_required_columns"] < summary_subject["expected_files"])
    ]
)

===== SUMMARY PER ACTIVITY & SUBJECT =====


,activity,subject,expected_files,found_files,readable_files,empty_files,files_with_required_columns,total_rows,missing_files,completion_percent
0,Bungkuk,Adelia,9,9,9,0,9,10475241,0,100.0
1,Bungkuk,Afi,9,9,9,0,9,10563945,0,100.0
2,Bungkuk,Aswangga,9,9,9,0,9,10431177,0,100.0
3,Bungkuk,Bustan,9,9,9,0,9,10030377,0,100.0
4,Bungkuk,Dilia,9,9,9,0,9,12014601,0,100.0
5,Bungkuk,Eldivo,9,9,9,0,9,11176425,0,100.0
6,Bungkuk,Fathir,9,9,9,0,9,10304265,0,100.0
7,Bungkuk,Lina,9,9,9,0,9,11184201,0,100.0
8,Bungkuk,Manda,9,9,9,0,9,11324841,0,100.0
9,Bungkuk,Miftah,9,9,9,0,9,10510473,0,100.0



Subject/activity yang belum lengkap:


,activity,subject,expected_files,found_files,readable_files,empty_files,files_with_required_columns,total_rows,missing_files,completion_percent


In [9]:
# =========================
# SAVE AUDIT REPORTS
# =========================

REPORT_DIR = BASE_DIR / "_audit_reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

detail_report_path = REPORT_DIR / "audit_detail_dataset_completeness.csv"
summary_report_path = REPORT_DIR / "audit_summary_by_activity_subject.csv"
missing_report_path = REPORT_DIR / "audit_missing_or_problem_files.csv"

merged_df.to_csv(detail_report_path, index=False)
summary_subject.to_csv(summary_report_path, index=False)
problem_df.to_csv(missing_report_path, index=False)

print("===== AUDIT REPORT SAVED =====")
print("Detail report :", detail_report_path)
print("Summary report:", summary_report_path)
print("Problem report:", missing_report_path)

print("\nFinal status:")
print("Expected files:", expected_total)
print("Found files   :", int(merged_df["file_exists"].sum()))
print("Missing files :", int(expected_total - merged_df["file_exists"].sum()))
print("Readable files:", int(merged_df["can_read"].sum()))
print("Problem files :", len(problem_df))

===== AUDIT REPORT SAVED =====
Detail report : /media/spell/Spell-lab/Lidar/A.Raw Dataset/_audit_reports/audit_detail_dataset_completeness.csv
Summary report: /media/spell/Spell-lab/Lidar/A.Raw Dataset/_audit_reports/audit_summary_by_activity_subject.csv
Problem report: /media/spell/Spell-lab/Lidar/A.Raw Dataset/_audit_reports/audit_missing_or_problem_files.csv

Final status:
Expected files: 432
Found files   : 432
Missing files : 0
Readable files: 432
Problem files : 0


Harusnya 

Expected total CSV = 432

Found files = harusnya 432

Missing files = harusnya 0

Readable files = harusnya 432

Files with required columns = harusnya 432

Problem files = harusnya 0